In [0]:
dbutils.widgets.text("uploaded_patient_ids_csv", "/Volumes/7_outgoing/pharos/upload/pharos_pilot_list.csv", "Accession number table")
dbutils.widgets.text("patient_ids_and_accession_numbers_table", "`7_outgoing`.pharos.pacs_accession_number_20260619", "Accession number table")
dbutils.widgets.text("sectra_dicom_folder_path", "/Volumes/7_outgoing/pharos/imaging/pharos_20260619/", "DICOM folder before anonymisation")
dbutils.widgets.text("llm_verifier_result_folder_path", "/Volumes/1_inland/sectra/vone/20260626_131626_llm_verifier2.parquet_batch", "llm verifier result parquets")

In [0]:
from pyspark.sql.functions import lit, col, from_json, schema_of_json, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

In [0]:


# DataFrame containing DICOM file paths
files_df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .option("includeBinaryFiles", "false")
    .load(dbutils.widgets.get("sectra_dicom_folder_path"))
    .select("path", "length")
)

from pyspark.sql.functions import expr
ROOT_DIR = "/Volumes/7_outgoing/pharos/imaging/pharos_20260619"
files_df = files_df.withColumn("full_path", expr("substring(path, 6, length(path))"))
files_df = files_df.withColumn("path", regexp_replace("path", lit(ROOT_DIR), ""))

from pyspark.sql.functions import regexp_extract

files_df = files_df.withColumn(
    "subdir",
    regexp_extract("path", r"(/?\d{6}/\d{6}/)", 1)
)

# Output schema
schema = StructType([
    StructField("full_path", StringType(), True),
    StructField("path", StringType(), True),
    StructField("subdir", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("study_datetime", TimestampType(), True),
    StructField("modality", StringType(), True),
    StructField("study_description", StringType(), True),
])

# mapInPandas function
def extract_dicom_tags(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for pdf in iterator:
        results = []

        for _, row in pdf.iterrows():
            full_path = row["full_path"]
            path = row["path"]
            subdir = row["subdir"]

            try:
                ds = pydicom.dcmread(
                    full_path,
                    stop_before_pixels=True,
                    force=True
                )

                accession_number = str(ds.get("AccessionNumber", "")) or None
                modality = str(ds.get("Modality", "")) or None
                study_description = str(ds.get("StudyDescription", "")) or None

                # Build Timestamp from StudyDate + StudyTime
                study_datetime = None
                study_date = str(ds.get("StudyDate", "")).strip()
                study_time = str(ds.get("StudyTime", "")).split(".")[0].strip()

                if study_date:
                    try:
                        if study_time:
                            dt_str = study_date + study_time.ljust(6, "0")
                            study_datetime = datetime.strptime(
                                dt_str, "%Y%m%d%H%M%S"
                            )
                        else:
                            study_datetime = datetime.strptime(
                                study_date, "%Y%m%d"
                            )
                    except Exception:
                        study_datetime = None

                results.append({
                    "full_path": full_path,
                    "path": path,
                    "subdir": subdir,
                    "accession_number": accession_number,
                    "study_datetime": study_datetime,
                    "modality": modality,
                    "study_description": study_description,
                })

            except (InvalidDicomError, FileNotFoundError, Exception):
                results.append({
                    "full_path": full_path,
                    "path": path,
                    "subdir": subdir,
                    "accession_number": None,
                    "study_datetime": None,
                    "modality": None,
                    "study_description": None,
                })

        yield pd.DataFrame(results)

# Execute extraction
dicom_tags_df = files_df.mapInPandas(
    extract_dicom_tags,
    schema=schema
)

output_path = "/Volumes/7_outgoing/pharos/imaging/pharos_20260619_metadata.parquet"
dicom_tags_df.write.mode("overwrite").parquet(output_path)

In [0]:
dt_df = spark.read.parquet(output_path)
ROOT_DIR = "/Volumes/7_outgoing/pharos/imaging/pharos_20260619"
from pyspark.sql.functions import regexp_replace, lit

#dt_df = dt_df.withColumn("path", regexp_replace("path", lit(ROOT_DIR), ""))
from pyspark.sql.functions import max

dt_df = dt_df.groupBy("subdir").agg(max("accession_number").alias("AN"), max("study_datetime").alias("study_datetime"))
display(dt_df.limit(1000))

In [0]:
pid_csv_path = dbutils.widgets.get("uploaded_patient_ids_csv")
pid_df = spark.read.option("header", "true").csv(pid_csv_path).select("research_id").distinct()
display(pid_df.limit(1000))

In [0]:
pid_acnbr_tab = dbutils.widgets.get("patient_ids_and_accession_numbers_table")
pid_acnbr_df = spark.read.format("delta").table(pid_acnbr_tab).distinct()
from pyspark.sql.functions import col

pid_acnbr_df = pid_acnbr_df.withColumnRenamed("Accession_Number", "accession_number")
display(pid_acnbr_df.limit(1000))

In [0]:
joined_df = pid_df.distinct().join(pid_acnbr_df.distinct(), pid_df.research_id == pid_acnbr_df.person_id, "left")
#display(joined_df.limit(1000))
#display(dt_df.limit(1000))
joined_df = joined_df.join(dt_df.distinct(), joined_df.accession_number == dt_df.AN, "left")
from pyspark.sql.functions import lit

#joined_df = joined_df.withColumn("llm_pii_detected", lit(None).cast("string"))
joined_df = joined_df.select("research_id", "accession_number", "subdir", "study_datetime")#, "llm_pii_detected")
#joined_df.write.format("delta").mode("overwrite").saveAsTable("7_outgoing.pharos.pharos_20260619_merged_record")
display(joined_df.limit(1000))


In [0]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import from_json, schema_of_json
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.functions import regexp_extract


llm_df = spark.read.parquet(*[f"/Volumes/1_inland/sectra/vone/20260626_131626_llm_verifier2.parquet_batch{idx}.parquet" for idx in range(14)])
llm_df = llm_df.withColumn(
    "path",
    regexp_replace(
        regexp_replace("path", "/Volumes/8_dev/pacs/anon_images/20260626_131626", ""),
        "_anon",
        ""
    )
)

llm_df = llm_df.withColumn(
    "subdir",
    regexp_extract("path", r"(/?\d{6}/\d{6}/)", 1)
)
sample_json = llm_df.select("llm_result").filter(col("llm_result").isNotNull()).limit(1).collect()[0]["llm_result"]
json_schema = schema_of_json(sample_json)
llm_df = llm_df.withColumn(
    "llm_result_json", from_json(col("llm_result"), json_schema)
).withColumn(
    "is_text_detected", col("llm_result_json.is_text_detected")
).withColumn(
    "is_personal_information_detected", col("llm_result_json.is_personal_information_detected")
)
llm_df = llm_df.select("subdir", "path", "is_personal_information_detected").filter(col("is_personal_information_detected") == "yes")
#llm_df = llm_df.join(joined_df.select("accession_number", "path"), on="path", how="left")


In [0]:
display(llm_df.filter("is_personal_information_detected == 'yes'").limit(100))

In [0]:
from pyspark.sql.functions import collect_set, col

agg_df = (
    llm_df.filter(col("is_personal_information_detected") == "yes")
    .groupBy("subdir")
    .agg(collect_set("path").alias("pii_detected_by_llm"))
)
display(agg_df)

In [0]:
joined_with_agg_df = joined_df.join(agg_df, on="subdir", how="left")
display(joined_with_agg_df.limit(1000))